# Task 10 Solutions — File Handling, Serialization & Recursion

This notebook works through the Task 10 sheet with complete, runnable solutions. Every question is repeated in its markdown cell, and under it you will find a full implementation with the actual output printed — so you can compare your own attempt line by line, understand *why* each solution works, and reuse the pattern in your own code. The first five problems are a tour of file I/O and serialization. Q1 streams a file and returns its final line; Q2 counts vowels across a text file into a dictionary; Q3 reads a tab-separated two-column file, adds a product column and a running `Total`; Q4 reverses each line's characters while carefully keeping the newline at the end; and Q5 counts word frequencies, serialises the dictionary to JSON with `json.dump`, reloads it with `json.load`, and answers queries from a given word list.

The last five problems are recursion drills. Q6 measures string length with no `len()`; Q7 computes the GCD with the Euclidean algorithm and not a single loop; Q8 implements the Levenshtein edit distance by following the given recurrence exactly; Q9 compresses runs of repeated characters into `[value, count]` pairs; and Q10 converts decimal to binary by stringing remainders together. Each solution is deliberately minimal — a handful of lines — which is the mark of good recursion: a base case that stops, and a recursive step that shrinks the problem.

Work the sheet in order. First attempt each problem yourself in the task notebook, then come here to validate. Where the solution teaches a technique (the streaming `last_line` trick, the `freq.get(word, 0) + 1` idiom, the newline-aware reversal), the theory cell explains both the idea and the exact decision the code makes.

### `Q-1`: Write a function `get_final_line(filename)`, which takes filename as input and return final line of the file.

Note: You can choose any file of your choice.

### Q1 — the final line of a file

The solution uses the **streaming idiom**: keep overwriting a `last_line` variable as you loop over the file, so when the loop ends it holds the final line.

```python
def get_final_line(filename):
    last_line = ''
    with open(filename, 'r') as f:
        for line in f:
            last_line = line
    return last_line
```

Why this shape? `for line in f` reads the file lazily one line at a time, so memory use stays flat even for huge files, and `''` is a safe initial value — for an empty file the function returns an empty string instead of crashing on `lines[-1]`. The `with` block guarantees the file closes automatically. The wrapper writes a two-line `sample.txt` (`line 1\nhello world\n`) and the function returns the final line `hello world`, printed with its trailing newline. Compare this with a `readlines()`-based answer: both are correct, but the streaming version scales, and it is the one to quote in interviews.

In [ ]:
# Write code here
def get_final_line(filename):
    last_line = ''
    with open(filename, 'r') as f:
        for line in f:
            last_line = line
    return last_line

with open('sample.txt', 'w') as f:
    f.write('line 1\nhello world\n')

print(get_final_line('sample.txt'))


hello world


### `Q-2`: Read through a text file, line by line. Use a dict to keep track of how many times each vowel (a, e, i, o, and u) appears in the file. Print the resulting tabulation -- dictionary.

### Q2 — vowel tabulation

Counting vowels is dictionary work: keys are the five vowels, values are running totals.

```python
def count_vowels(filename):
    vowels = {'a': 0, 'e': 0, 'i': 0, 'o': 0, 'u': 0}
    with open(filename, 'r') as f:
        text = f.read().lower()
    for ch in text:
        if ch in vowels:
            vowels[ch] += 1
    return vowels
```

Two decisions matter. First, `f.read().lower()` loads the whole file and lower-cases it in one step, so `Alice` and `alice` both land in the `'a'` bucket — without this, uppercase vowels would be missed. Second, the membership test `if ch in vowels` doubles as the vowel check: the five keys are exactly the five vowels, so punctuation and consonants are skipped automatically. The output `{'a': 0, 'e': 2, 'i': 1, 'o': 2, 'u': 0}` reflects the file's content — `hello world` has two `e`'s, one `i`, two `o`'s, and none of the rest. Note the loop over the whole text is a simple, readable choice; for very large files you would stream line by line instead.

In [ ]:
# write code here
def count_vowels(filename):
    vowels = {'a': 0, 'e': 0, 'i': 0, 'o': 0, 'u': 0}
    with open(filename, 'r') as f:
        text = f.read().lower()
    for ch in text:
        if ch in vowels:
            vowels[ch] += 1
    return vowels

print(count_vowels('sample.txt'))


{'a': 0, 'e': 2, 'i': 1, 'o': 2, 'u': 0}


### `Q-3`: Create a text file (using an editor, not necessarily Python) containing two tab separated columns, with each column containing a number. Then use Python to read through the file you've created. For each line, multiply each first number by the second and include it in the file in third column. In last add a line Total, by summing the value of third column

Input File example:

```
1   2
3   4
5   6
7   8
9   10
```

Output File Example:

```
1   2   2
3   4   12
5   6   30
7   8   56
9   10  90
Total   190
```

### Q3 — tab-separated arithmetic with a Total row

The solution builds the input file first, then performs a read–transform–write cycle.

```python
with open('numbers.txt', 'w') as f:
    f.write('1\t2\n3\t4\n...')

with open('numbers.txt', 'r') as f:
    lines = f.readlines()

rows = []
total = 0
for line in lines:
    a, b = line.strip().split('\t')
    product = int(a) * int(b)
    total += product
    rows.append(f"{a}\t{b}\t{product}\n")
rows.append(f"Total\t{total}\n")

with open('numbers.txt', 'w') as f:
    f.writelines(rows)
```

Walk the pipeline: `strip()` removes the trailing newline so `split('\t')` yields exactly two clean numbers; `int()` converts them before multiplication; each transformed row keeps the tabs so the columns align, and gains a third tab-separated product. The grand total accumulates across the loop and is written last as the `Total` row. The expected output confirms the arithmetic: products `2, 12, 30, 56, 90` and `Total 190`. Rewriting `numbers.txt` with `'w'` is safe because we finished reading first; the final `writelines(rows)` writes the whole result in one call.

In [ ]:
# write code here
# creating the input file (tab separated)
with open('numbers.txt', 'w') as f:
    f.write('1\t2\n3\t4\n5\t6\n7\t8\n9\t10\n')

# reading the file
with open('numbers.txt', 'r') as f:
    lines = f.readlines()

rows = []
total = 0
for line in lines:
    a, b = line.strip().split('\t')
    product = int(a) * int(b)
    total += product
    rows.append(f"{a}\t{b}\t{product}\n")
rows.append(f"Total\t{total}\n")

# writing back to the file with third column + Total line
with open('numbers.txt', 'w') as f:
    f.writelines(rows)

print(''.join(rows))


1	2	2
3	4	12
5	6	30
7	8	56
9	10	90
Total	190


### `Q-4`: Create line wise reverse of a file
Write a function which takes two arguments: the names of the input file (to be read from) and the output file (which will be created).

For example, if a file looks like

```
abc def
ghi jkl
```

then the output file will be

```
fed cba
lkj ihg
```

**Notice**: The newline remains at the end of the string, while the rest of the characters are all reversed.

### Q4 — line-wise reverse, newline intact

The trap in this problem is the newline. If you just reverse the whole line, `'abc def\n'` becomes `'\nfed cba'` and the file is mangled. The solution separates the newline from the content first:

```python
def reverse_file(input_file, output_file):
    with open(input_file, 'r') as f:
        lines = f.readlines()
    with open(output_file, 'w') as f:
        for line in lines:
            newline = ''
            if line.endswith('\n'):
                line = line[:-1]
                newline = '\n'
            f.write(line[::-1] + newline)
```

For every line, `endswith('\n')` detects the trailing newline; if present, slicing off the last character (`line[:-1]`) removes it and we remember to re-add `'\n'`. Only then do we reverse the content with `line[::-1]` (a full reverse slice) and write it back with the newline restored. The test generates `data.txt` with `abc def\nghi jkl\n` and produces `fed cba\nlkj ihg\n` — characters reversed, line structure untouched. This newline-aware pattern is exactly what the question's warning is about, and it is the detail to remember.

In [ ]:
# write code here
def reverse_file(input_file, output_file):
    with open(input_file, 'r') as f:
        lines = f.readlines()

    with open(output_file, 'w') as f:
        for line in lines:
            newline = ''
            if line.endswith('\n'):
                line = line[:-1]
                newline = '\n'
            f.write(line[::-1] + newline)

with open('data.txt', 'w') as f:
    f.write('abc def\nghi jkl\n')

reverse_file('data.txt', 'data_rev.txt')

with open('data_rev.txt', 'r') as f:
    print(f.read(), end='')


fed cba
lkj ihg


### `Q-5`: Create a Serialized dict of frequency of words in the file. And from given list of words, using serialized dict show word count.

* List of word will be given

Given String
```python
strings = """Alice was beginning to get very tired of sitting by her sister
            on the bank, and of having nothing to do:  once or twice she had
            peeped into the book her sister was reading, but it had no
            pictures or conversations in it, `and what is the use of a book,'
            thought Alice `without pictures or conversation?'

            So she was considering in her own mind (as well as she could,
            for the hot day made her feel very sleepy and stupid), whether
            the pleasure of making a daisy-chain would be worth the trouble
            of getting up and picking the daisies, when suddenly a White
            Rabbit with pink eyes ran close by her.

            There was nothing so VERY remarkable in that; nor did Alice
            think it so VERY much out of the way to hear the Rabbit say to
            itself, `Oh dear!  Oh dear!  I shall be late!'  (when she thought
            it over afterwards, it occurred to her that she ought to have
            wondered at this, but at the time it all seemed quite natural);
            but when the Rabbit actually TOOK A WATCH OUT OF ITS WAISTCOAT-
            POCKET, and looked at it, and then hurried on, Alice started to
            her feet, for it flashed across her mind that she had never
            before seen a rabbit with either a waistcoat-pocket, or a watch to
            take out of it, and burning with curiosity, she ran across the
            field after it, and fortunately was just in time to see it pop
            down a large rabbit-hole under the hedge."""

word_list = ['alice', 'wonder', 'natural']
```

### Q5 — word frequency serialised to JSON

This solution ties file reading, text cleaning, and JSON serialization into one pipeline.

```python
freq = {}
for word in strings.lower().split():
    word = word.strip('.,()!?;:\'"`-')  # strip two rows of punctuation
    if word:
        freq[word] = freq.get(word, 0) + 1

with open('word_freq.json', 'w') as f:
    json.dump(freq, f)
with open('word_freq.json', 'r') as f:
    freq = json.load(f)

for w in word_list:
    print(w, ':', freq.get(w, 0))
```

`lower().split()` normalises case and breaks the text on whitespace; the `strip` call removes surrounding punctuation so `'Alice'` and `'alice,'` collapse into the same token; `freq.get(word, 0) + 1` is the one-line idiom for incrementing a counter that may not exist yet. We then **serialise** with `json.dump` and **deserialise** with `json.load`, proving the round trip. The queries come from the reloaded dict: `alice : 4`, `wonder : 0` (missing, so `.get` returns 0), `natural : 1`. Serialising to disk means the heavy counting happens once; every later lookup reads the JSON file directly.

In [ ]:
# write code here
import json

strings = """Alice was beginning to get very tired of sitting by her sister
            on the bank, and of having nothing to do:  once or twice she had
            peeped into the book her sister was reading, but it had no
            pictures or conversations in it, `and what is the use of a book,'
            thought Alice `without pictures or conversation?'

            So she was considering in her own mind (as well as she could,
            for the hot day made her feel very sleepy and stupid), whether
            the pleasure of making a daisy-chain would be worth the trouble
            of getting up and picking the daisies, when suddenly a White
            Rabbit with pink eyes ran close by her.

            There was nothing so VERY remarkable in that; nor did Alice
            think it so VERY much out of the way to hear the Rabbit say to
            itself, `Oh dear!  Oh dear!  I shall be late!'  (when she thought
            it over afterwards, it occurred to her that she ought to have
            wondered at this, but at the time it all seemed quite natural);
            but when the Rabbit actually TOOK A WATCH OUT OF ITS WAISTCOAT-
            POCKET, and looked at it, and then hurried on, Alice started to
            her feet, for it flashed across her mind that she had never
            before seen a rabbit with either a waistcoat-pocket, or a watch to
            take out of it, and burning with curiosity, she ran across the
            field after it, and fortunately was just in time to see it pop
            down a large rabbit-hole under the hedge."""

word_list = ['alice', 'wonder', 'natural']

# building the word frequency dict
freq = {}
for word in strings.lower().split():
    word = word.strip('.,()!?;:\'"`-')
    if word:
        freq[word] = freq.get(word, 0) + 1

# serializing (dump) to a json file
with open('word_freq.json', 'w') as f:
    json.dump(freq, f)

# deserializing (load) and showing counts for the given list
with open('word_freq.json', 'r') as f:
    freq = json.load(f)

for w in word_list:
    print(w, ':', freq.get(w, 0))


alice : 4
wonder : 0
natural : 1


### `Q-6`: Given a string calculate length of the string using recursion.

**Example 1:**

Input:
```bash
"abcd"
```
Output:
```bash
4
```

**Example 2:**

Input:
```bash
DataScience
```
Output:
```bash
11
```

### Q6 — recursive string length

The definition of length is naturally recursive: an empty string has length `0`; any other string is `1` longer than itself minus its first character.

```python
def str_length(s):
    if s == '':
        return 0
    return 1 + str_length(s[1:])
```

The base case is the empty string. The recursive step slices off the first character with `s[1:]` and adds `1` for it — the recursion peels one character per call until nothing remains, then unwinds adding `1` at every level. `str_length('abcd')` becomes `1 + 1 + 1 + 1 + 0 = 4`, and `'DataScience'` (11 characters) returns `11`. The elegance is the two-line body: no loops, no `len()`, just the base case and the self-call. Watch that each call creates `s[1:]`, so memory is O(n); Python's recursion limit (about 1000 frames) is far beyond any string you'd test here.

In [ ]:
# Write code here
def str_length(s):
    if s == '':
        return 0
    return 1 + str_length(s[1:])

print(str_length('abcd'))
print(str_length('DataScience'))


4
11


### `Q-7`: Write a function that accepts two numbers and returns their greatest common divisior. Without using any loop

`def gcd(int, int) => int`

```
gcd(16,24) will give 8
```

### Q7 — recursive GCD without loops

Euclid's algorithm has a perfect recursive form: `gcd(a, b)` equals `a` when `b` is `0`, otherwise it equals `gcd(b, a % b)`.

```python
def gcd(a, b):
    if b == 0:
        return a
    return gcd(b, a % b)
```

Why replace `a` with `a % b`? Any common divisor of `a` and `b` also divides `a % b`, so the GCD survives the replacement while the numbers shrink — the remainder is always smaller than `b`, guaranteeing the recursion reaches `b == 0`. Tracing `gcd(16, 24)`: `gcd(24, 16)` → `gcd(16, 8)` → `gcd(8, 0)` → `8`. The function is three lines, has no loop anywhere, and returns the exact answer the question demands (`gcd(16, 24)` prints `8`). This is the canonical "no loops" recursion interview solution, so memorise the shape: base case `b == 0` returning `a`, recursive case calling `gcd(b, a % b)`.

In [ ]:
# Write code here
def gcd(a, b):
    if b == 0:
        return a
    return gcd(b, a % b)

print(gcd(16, 24))


8


### `Q-8`: String Edit Distance

Use your recursive function to write a program that reads two strings from the user and displays the edit distance between them.

*The edit distance between two strings is a measure of their similarity - the smaller the edit distance, the more similar the strings are with regard to the minimum number of insert, delete and substitute operations needed to transform one string into the other.*

Consider the strings `kitten` and `sitting`. The first string can be transformed into the second with: substitute `k` with `s`, substitute `e` with `i`, and insert a `g` at the end. Edit distance is `3`.

Algorithm:
```
Let s and t be the strings
    If the length of s is 0 then
        Return the length of t
    Else if the length of t is 0 then
        Return the length of s
    Else
        Set cost to 0
        If the last character in s does not equal the last character in t then
            Set cost to 1
        Set d1 equal to the edit distance between all characters except the last one in s, and all characters in t, plus 1
        Set d2 equal to the edit distance between all characters in s, and all characters except the last one in t, plus 1
        Set d3 equal to the edit distance between all characters except the last one in s, and all characters except the last one in t, plus cost
        Return the minimum of d1, d2 and d3
```

### Q8 — string edit distance

The solution transcribes the given algorithm into code literally, and that is the intended difficulty: recursion on two strings that shrink from the end.

```python
def edit_distance(s, t):
    if len(s) == 0:
        return len(t)
    elif len(t) == 0:
        return len(s)
    else:
        cost = 0
        if s[-1] != t[-1]:
            cost = 1
        d1 = edit_distance(s[:-1], t) + 1
        d2 = edit_distance(s, t[:-1]) + 1
        d3 = edit_distance(s[:-1], t[:-1]) + cost
        return min(d1, d2, d3)
```

The base cases are the edges where one string is empty — the remaining operations are the length of the other. Otherwise we compare the **last** characters: `cost` is `0` if they match, `1` if a substitution would be needed. `d1` deletes `s`'s last character, `d2` deletes `t`'s, and `d3` aligns them paying `cost`. The answer is the cheapest of the three paths. For `'kitten'` and `'sitting'` this evaluates to `3` — substitute `k→s`, `e→i`, insert `g`. The naive version is exponential, which is fine for short strings and exactly what the lesson wants you to see.

In [ ]:
# write code here
def edit_distance(s, t):
    if len(s) == 0:
        return len(t)
    elif len(t) == 0:
        return len(s)
    else:
        cost = 0
        if s[-1] != t[-1]:
            cost = 1
        d1 = edit_distance(s[:-1], t) + 1
        d2 = edit_distance(s, t[:-1]) + 1
        d3 = edit_distance(s[:-1], t[:-1]) + cost
        return min(d1, d2, d3)

print("Edit distance:", edit_distance('kitten', 'sitting'))


Edit distance: 3


### `Q-9`: Run-Length Encoding

Run-length encoding is a simple data compression technique that can be effective when repeated values occur at adjacent positions within a list. Compression is achieved by replacing groups of repeated values with one copy of the value, followed by the number of times that the value should be repeated. For example, the list
```
["A", "A", "A", "A", "A", "A", "A", "A", "A", "A", "A", "A", "B", "B", "B", "B", "A", "A", "A", "A", "A", "A", "B"]
```
would be compressed as `["A", 12, "B", 4, "A", 6, "B", 1]`.

Write a recursive function that implements the run-length compression technique described above. Your function will take a list or a string as its only parameter. It should return the run-length compressed list as its only result. Include a main program that reads a string from the user, compresses it, and displays the run-length encoded result.

### Q9 — run-length encoding

Run-length encoding compresses each run of identical values into one copy of the value plus its run length. The recursive function encodes the *first* run and recurses on what remains:

```python
def run_length_encode(data):
    if not data:
        return []
    count = 1
    rest = data[1:]
    while rest and rest[0] == data[0]:
        count += 1
        rest = rest[1:]
    return [data[0], count] + run_length_encode(rest)
```

The base case is the empty input → `[]`. Otherwise we count how many leading elements equal `data[0]` by trimming `rest` until the first different element (the `while` looks ahead without touching the original), then emit the pair `[data[0], count]` and append the recursion on the untouched tail. `'AAAAAAAAAAAABBBBAAAAAAB'` becomes `['A', 12, 'B', 4, 'A', 6, 'B', 1]` — twelve A's, four B's, six A's, one B — and `'hello'` becomes `['h', 1, 'e', 1, 'l', 2, 'o', 1]`, showing that even unique letters still get encoded. The function accepts lists or strings because both support indexing and slicing.

In [ ]:
# write code here
def run_length_encode(data):
    if not data:
        return []
    count = 1
    rest = data[1:]
    while rest and rest[0] == data[0]:
        count += 1
        rest = rest[1:]
    return [data[0], count] + run_length_encode(rest)

print(run_length_encode('AAAAAAAAAAAA' + 'BBBB' + 'AAAAAA' + 'B'))
print(run_length_encode('hello'))


['A', 12, 'B', 4, 'A', 6, 'B', 1]
['h', 1, 'e', 1, 'l', 2, 'o', 1]


### `Q-10`: Write a recursive function to convert a decimal to binary

### Q10 — decimal to binary by recursion

Converting to binary is repeated division by 2, collecting remainders from last to first. Recursion gives the backwards order for free, because the recursive call happens *before* the current digit is appended:

```python
def dec_to_bin(n):
    if n == 0:
        return '0'
    elif n == 1:
        return '1'
    else:
        return dec_to_bin(n // 2) + str(n % 2)
```

The base cases cover the digits `0` and `1`. For anything larger, `dec_to_bin(n // 2)` computes all the *more significant* digits first, and `str(n % 2)` tacks the current remainder onto the end. Tracing `dec_to_bin(10)`: `bin(5)+'0'` → `(bin(2)+'1')+'0'` → `((bin(1)+'0')+'1')+'0'` → `'1010'`. And `25` → `11001`. Because the deepest call produces the leading digit, the string assembles in the correct order with no reversing step — a neat property of calling yourself before appending.

## Summary

This solutions sheet completed every Task 10 problem. On file handling and serialization you: streamed a file to grab its final line, tabulated vowels with a dictionary, added a product column and running `Total` to a tab-separated file, reversed lines while preserving their newlines, and serialised a word-frequency dictionary to JSON and queried it back. On recursion you re-implemented string length, applied the Euclidean algorithm for GCD, followed the edit-distance recurrence exactly, compressed runs with run-length encoding, and built decimal-to-binary conversion. Throughout, the pattern to remember is twofold: file problems reward reading carefully (strip, split, newlines, `with`), and recursion problems reward a crisp base case plus a shrinking recursive step. Re-code any question you had to look at — the twenty lines across these ten solutions are core interview material.

In [ ]:
# Write code here
def dec_to_bin(n):
    if n == 0:
        return '0'
    elif n == 1:
        return '1'
    else:
        return dec_to_bin(n // 2) + str(n % 2)

print(dec_to_bin(10))
print(dec_to_bin(25))


1010
11001
